In [9]:
# ===========================================
# Comprehensive BEBs Morphostate Index and Morphotype Notebook
# ===========================================

# Step 0: Import libraries
import os
import pandas as pd
import numpy as np

# Set working directory where the Excel file is stored
directory = r"C:\Users\mgaz4900\OneDrive - The University of Sydney (Staff)\Chapter-4 Predicting BEBs Morphosates\Codes\BEBs Morphostate Index"
os.chdir(directory)

# Step 1: Load data

df = pd.read_excel(
    "Data_BEBs.xlsx",
    sheet_name='data_application'
)

# Preview first rows to confirm data loaded correctly
df.head()

# ===========================================
# Step 2: Define index calculation functions
# ===========================================

# ------------------------
# 2a. Parameter A: Swell–Entrance Alignment (SEA)
# ------------------------
# Measures alignment between dominant swell direction and estuary/bay entrance
# High values (~1) indicate well-aligned entrance-swell orientation
def compute_Parameter_A(df):
    # Calculate difference in degrees
    df["Delta_theta1"] = abs(df["Swell Direction (SD)"] - df["Entrance Orientation (EO)"])
    # Adjust differences greater than 180 degrees
    df["Delta_theta1"] = np.where(df["Delta_theta1"] > 180, 360 - df["Delta_theta1"], df["Delta_theta1"])
    # Compute SEAI normalized between 0–1
    df["SEA"] = (1 + np.cos(np.radians(df["Delta_theta1"]))) / 2
    df["SEA"] = df["SEA"].round(2)
    
    # Classification for interpretation
    def classify_sea(val):
        if val < 0.35: return "Poor Alignment"
        elif val <= 0.6: return "Moderate Alignment"
        else: return "Good Alignment"
    
    df["SEAIm_Class"] = df["SEA"].apply(classify_sea)
    return df

# ------------------------
# 2b. Parameter B: Beach-Entrance Alignment (BEA)
# ------------------------
# Measures alignment between entrance direction and beach orientation
def compute_Parameter_B(df):
    df["Delta_theta2"] = abs(df["Entrance Orientation (EO)"] - df["Beach Orientation (BO)"])
    df["Delta_theta2"] = np.where(df["Delta_theta2"] > 180, 360 - df["Delta_theta2"], df["Delta_theta2"])
    df["BEA"] = (1 + np.cos(np.radians(df["Delta_theta2"]))) / 2
    df["BEA"] = df["BEA"].round(2)
    
    def classify_bea(val):
        if val < 0.35: return "Poor Alignment"
        elif val <= 0.6: return "Moderate Alignment"
        else: return "Good Alignment"
    
    df["EBAI_Class"] = df["BEA"].apply(classify_bea)
    return df

# ------------------------
# 2c. Parameter C: Integrated Coastal Sheltering (ICS)
# ------------------------
# Combines coastline straightness (CSI) and local barrier index (LBI)
def compute_Parameter_C(df):
    df["ICS_RAW"] = df["Actual Length (AL) (m)"] / df["Straight Length (SL) (m)"]
    ICS_min = 1
    ICS_max = 4
    df["ICS_normalized"] = (df["ICS_RAW"] - ICS_min) / (ICS_max - ICS_min)
    # Barrier effectiveness normalized 0–1
    df["LBI"] = df["Barrier Effectiveness"].round(2)
    df["ICS"] = ((df["LBI"] + df["ICS_normalized"]) / 2).round(2)
    return df


# ------------------------
# 2d. Parameter D: Mean Depth (MD)
# ------------------------
# Normalized mean depth
def compute_Parameter_D(df):
    ED_min = 2
    ED_max = 60
    df["MD"] = (df["Embayment Depth (ED) (m)"] - ED_min) / (ED_max - ED_min)
    return df


# ------------------------
# 2e. Parameter E: Entrance Width-to-Area Ratio (EWAR)
# ------------------------
# Indicates entrance openness relative to bay size
def compute_Parameter_E(df):
    EWAR_min = 0.04
    EWAR_max = 4.2
    df["EWARr"] = df["Entrance Width (EW) (km)"] / (df["Bay/Estuary Area (BA) (km2)"] ** 0.5)
    df["EWAR"] = (df["EWARr"] - EWAR_min) / (EWAR_max - EWAR_min)
    df["EWAR"] = df["EWAR"].round(2)
    
    def categorize_ewar(val):
        if val > 0.15: return "Wide"
        elif val > 0.10: return "Medium"
        elif val > 0.02: return "Narrow"
        else: return "Restricted"
    
    df["Entrance_Type"] = df["EWAR"].apply(categorize_ewar)
    return df


# ------------------------
# 2f. Parameter F: Fetch Adjusted Distance from Entrance (FADE)
# ------------------------
# Quantifies exposure based on distance from entrance normalized by fetch
def compute_Parameter_F(df):
    df["FADE"] = (df["BEBs Distance from the Entrance (m)"] / df["Longest Fetch (m)"]).round(2)
    
    def classify_fade(val):
        if val < 0.3: return "Close to the Entrance"
        elif val <= 0.6: return "Mid Bay/Estuary relative to the Entrance"
        else: return "Furthest from the Entrance"
    
    df["FADE_Class"] = df["FADE"].apply(classify_fade)
    return df


# ------------------------
# 2g. Parameter G: Swell Potentiality (SP)
# ------------------------
# Measures potential wave energy impacting beach
def compute_Parameter_G(df):
    SP_min = 0.10
    SP_max = 3.67
    df["SP"] = (df["Swell Height (SH) (m)"] - SP_min) / (SP_max - SP_min)
    return df


# ------------------------
# 2h. Morphostate Index & Classification
# ------------------------
# Combines all indices to assign a beach morphotype
def compute_morphotypes_predictor(df):
    intercept = 0.4752
    df["Morphotypes_Predictor_Score"] = (
        df["SEA"] * 0.1487 +
        df["BEA"] * 0.5137 +
        df["EWAR"] * 0.0010 +
        df["FADE"] * -0.9271 +
        df["MD"] * 0.0054 +
        df["ICS"] * -0.1124 +
        df["SP"] * 0.0183
    ) + intercept
    df["Morphotypes_Predictor_Score"] = df["Morphotypes_Predictor_Score"].round(3)
    
    def classify_morphotypes(Ym):
        if Ym < 0.40: return "Mostly Concave to Concave"
        elif 0.40 <= Ym <= 0.65: return "Mostly Linear to Linear"
        elif Ym > 0.65: return "Mostly Convex to Convex"
        else: return "Unclassified"
    
    df["Morphotypes_Class"] = df["Morphotypes_Predictor_Score"].apply(classify_morphotypes)
    return df


# ===========================================
# Step 3: Apply all functions sequentially
# ===========================================
# Instructions: run all these sequentially to compute all indices and the final Morphostate Index
df = compute_Parameter_A(df)
df = compute_Parameter_B(df)
df = compute_Parameter_C(df)
df = compute_Parameter_D(df)
df = compute_Parameter_E(df)
df = compute_Parameter_F(df)
df = compute_Parameter_G(df)
df = compute_morphotypes_predictor(df)

# ===========================================
# Step 4: Display final results
# ===========================================
# Select columns of interest for final analysis

Results = ["BEBs","SEA","BEA","EWAR","FADE", "ICS","MD","SP","Morphotypes_Predictor_Score", "Morphotypes_Class"]

# Display final table with indices and classifications
df[Results]


,BEBs,SEA,BEA,EWAR,FADE,ICS,MD,SP,Morphotypes_Predictor_Score,Morphotypes_Class
0,Botafogo,0.67,0.67,0.01,0.11,0.63,0.103448,0.159664,0.750,Mostly Convex to Convex
1,Charitas,0.67,0.18,0.01,0.21,1.62,0.103448,0.159664,0.294,Mostly Concave to Concave
2,Maua,0.67,0.99,0.01,1.06,0.05,0.103448,0.159664,0.099,Mostly Concave to Concave
3,Bica,0.67,0.95,0.01,0.57,0.10,0.103448,0.159664,0.527,Mostly Linear to Linear
4,Grande,0.67,0.01,0.01,0.68,0.54,0.103448,0.159664,-0.108,Mostly Concave to Concave
5,Pringle Bay,0.96,0.41,0.23,0.20,0.19,0.655172,0.445378,0.634,Mostly Linear to Linear
6,Gordons Bay Main,0.96,0.50,0.23,0.91,0.23,0.655172,0.445378,0.017,Mostly Concave to Concave
7,Fish Hoek,0.96,0.71,0.23,0.95,0.51,0.655172,0.445378,0.057,Mostly Concave to Concave
8,Buffels Bay,0.96,0.59,0.23,0.22,0.41,0.655172,0.445378,0.683,Mostly Convex to Convex
9,Muizenberg,0.96,1.00,0.23,1.11,0.00,0.655172,0.445378,0.114,Mostly Concave to Concave
